# Flipkart Gridlock 2.0: V9 Spatial-Temporal Tweedie Architecture
## Walk-Forward Validation & Exponential Decay Implementation

**Architectural Paradigm:**
1. **Time-Series Validation**: Replaces standard K-Fold to strictly prevent chronological leakage.
2. **Tweedie Distribution**: Objective function tailored for zero-inflated target vectors with extreme positive right-tails.
3. **Hierarchical Spatial Flow**: Evaluates parent geohash aggregations to simulate neighborhood traffic bleed.
4. **Temporal Decay**: Applies exponential sample weighting to prioritize recent topological states.

In [1]:
import os
import warnings
import numpy as np
import pandas as pd
import pygeohash as pgh
from scipy.optimize import minimize
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import r2_score

import lightgbm as lgb
import xgboost as xgb

warnings.filterwarnings('ignore')
np.random.seed(42)

data_paths = [".", "data/raw", "../../data/raw"]
base_path = next((path for path in data_paths if os.path.exists(os.path.join(path, "train.csv"))), None)

raw_train = pd.read_csv(os.path.join(base_path, "train.csv"))
raw_test = pd.read_csv(os.path.join(base_path, "test.csv"))

# Sort chronologically to guarantee integrity of walk-forward validation
raw_train.sort_values(by=['day', 'timestamp'], inplace=True)
raw_train.reset_index(drop=True, inplace=True)

y_train = raw_train['demand'].values
submission_index = raw_test['Index'].values

In [2]:
def synthesize_base_features(df):
    df_features = df.copy()
    
    # Extract parent geohash to approximate adjacent neighborhood flow
    df_features['parent_geohash'] = df_features['geohash'].str[:-1]
    
    time_components = df_features['timestamp'].str.split(':', expand=True).astype(int)
    df_features['ts_minutes'] = time_components[0] * 60 + time_components[1]
    df_features['hour'] = time_components[0]
    
    # Continuous temporal encoding
    df_features['hour_sin'] = np.sin(2 * np.pi * df_features['hour'] / 24.0)
    df_features['hour_cos'] = np.cos(2 * np.pi * df_features['hour'] / 24.0)
    
    df_features['Temperature'] = df_features['Temperature'].fillna(df_features['Temperature'].median())
    df_features['Weather'] = df_features['Weather'].fillna('Unknown')
    
    return df_features

X_train_base = synthesize_base_features(raw_train.drop(columns=['demand'], errors='ignore'))
X_test_base = synthesize_base_features(raw_test)

In [3]:
def construct_temporal_lags(source_df, target_df):
    lag_intervals = {
        'lag_24h': 1,
        'lag_48h': 2,
        'lag_168h': 7
    }
    
    df_merged = target_df.copy()
    
    for col_name, day_offset in lag_intervals.items():
        lag_df = source_df[['geohash', 'day', 'timestamp', 'demand']].copy()
        lag_df['day'] += day_offset
        lag_df.rename(columns={'demand': col_name}, inplace=True)
        
        df_merged = df_merged.merge(lag_df, on=['geohash', 'day', 'timestamp'], how='left')
    
    # Strict imputation: Missing historical records denote absence of traffic, not the city mean
    lag_columns = list(lag_intervals.keys())
    df_merged[lag_columns] = df_merged[lag_columns].fillna(0.0)
    
    return df_merged

X_train_lagged = construct_temporal_lags(raw_train, X_train_base)
X_test_lagged = construct_temporal_lags(raw_train, X_test_base)

In [6]:
# Compute global mappings for base nodes and parent nodes
node_demand_map = raw_train.groupby('geohash')['demand'].mean().to_dict()
parent_demand_map = raw_train.groupby(X_train_base['parent_geohash'])['demand'].mean().to_dict()

X_train_lagged['node_historical_mean'] = X_train_lagged['geohash'].map(node_demand_map).fillna(0.0)
X_test_lagged['node_historical_mean'] = X_test_lagged['geohash'].map(node_demand_map).fillna(0.0)

X_train_lagged['neighborhood_flow_mean'] = X_train_lagged['parent_geohash'].map(parent_demand_map).fillna(0.0)
X_test_lagged['neighborhood_flow_mean'] = X_test_lagged['parent_geohash'].map(parent_demand_map).fillna(0.0)

categorical_features = ['geohash', 'parent_geohash', 'RoadType', 'Weather', 'LargeVehicles', 'Landmarks']

for category in categorical_features:
    encoder = LabelEncoder()
    # Cast to string safely handles any hidden NaNs before integer transformation
    combined_data = X_train_lagged[category].astype(str).tolist() + X_test_lagged[category].astype(str).tolist()
    encoder.fit(combined_data)
    
    X_train_lagged[category] = encoder.transform(X_train_lagged[category].astype(str))
    X_test_lagged[category] = encoder.transform(X_test_lagged[category].astype(str))

drop_columns = ['timestamp', 'Index']
training_features = [col for col in X_train_lagged.columns if col not in drop_columns]

X = X_train_lagged[training_features].values
X_test = X_test_lagged[training_features].values
temporal_indices = X_train_lagged['day'].values

print(f"Matrix successfully compiled. Training shape: {X.shape}")

Matrix successfully compiled. Training shape: (77299, 18)


In [7]:
tscv = TimeSeriesSplit(n_splits=5)

lgbm_params = {
    'objective': 'tweedie',
    'tweedie_variance_power': 1.5,
    'metric': 'rmse',
    'learning_rate': 0.03,
    'max_depth': 8,
    'min_child_samples': 20,
    'verbose': -1,
    'random_state': 42,
    'n_jobs': -1
}

xgb_params = {
    'objective': 'reg:tweedie',
    'tweedie_variance_power': 1.5,
    'eval_metric': 'rmse',
    'learning_rate': 0.03,
    'max_depth': 7,
    'random_state': 42,
    'n_jobs': -1
}

oof_predictions_lgb = np.zeros(len(X))
oof_predictions_xgb = np.zeros(len(X))
test_predictions_lgb = np.zeros(len(X_test))
test_predictions_xgb = np.zeros(len(X_test))

print("Executing Walk-Forward TimeSeries Validation...")

for fold, (train_idx, val_idx) in enumerate(tscv.split(X)):
    X_tr, y_tr, days_tr = X[train_idx], y_train[train_idx], temporal_indices[train_idx]
    X_va, y_va = X[val_idx], y_train[val_idx]
    
    # Exponential decay function to penalize stale observations
    max_day_current_fold = days_tr.max()
    decay_weights = np.exp((days_tr - max_day_current_fold) / 15.0)
    
    # LightGBM Engine
    lgb_train_data = lgb.Dataset(X_tr, y_tr, weight=decay_weights)
    lgb_val_data = lgb.Dataset(X_va, y_va)
    model_lgb = lgb.train(
        lgbm_params, 
        lgb_train_data, 
        num_boost_round=2500, 
        valid_sets=[lgb_val_data], 
        callbacks=[lgb.early_stopping(100, verbose=False)]
    )
    oof_predictions_lgb[val_idx] = model_lgb.predict(X_va)
    test_predictions_lgb += model_lgb.predict(X_test) / tscv.n_splits
    
    # XGBoost Engine
    xgb_train_matrix = xgb.DMatrix(X_tr, label=y_tr, weight=decay_weights)
    xgb_val_matrix = xgb.DMatrix(X_va, label=y_va)
    model_xgb = xgb.train(
        xgb_params, 
        xgb_train_matrix, 
        num_boost_round=2500, 
        evals=[(xgb_val_matrix, 'val')], 
        early_stopping_rounds=100, 
        verbose_eval=False
    )
    oof_predictions_xgb[val_idx] = model_xgb.predict(xgb_val_matrix)
    test_predictions_xgb += model_xgb.predict(xgb.DMatrix(X_test)) / tscv.n_splits
    
    print(f"Fold {fold+1} completed. Validation R2: {r2_score(y_va, oof_predictions_lgb[val_idx]):.4f}")

Executing Walk-Forward TimeSeries Validation...
Fold 1 completed. Validation R2: 0.8077
Fold 2 completed. Validation R2: 0.7371
Fold 3 completed. Validation R2: 0.8058
Fold 4 completed. Validation R2: 0.8588
Fold 5 completed. Validation R2: 0.8038


In [10]:
# Limit optimization space strictly to validated temporal folds
validated_indices = np.where(oof_predictions_lgb > 0)[0]

def optimization_objective(weights):
    weights = np.array(weights)
    if weights.sum() == 0:
        return 999.0
    normalized_weights = weights / weights.sum()
    blended_prediction = (normalized_weights[0] * oof_predictions_lgb[validated_indices]) + \
                         (normalized_weights[1] * oof_predictions_xgb[validated_indices])
    return -max(0, 100 * r2_score(y_train[validated_indices], blended_prediction))

optimal_weights = minimize(optimization_objective, [0.5, 0.5], method='Nelder-Mead').x
optimal_weights /= sum(optimal_weights)

final_test_predictions = np.clip(
    (optimal_weights[0] * test_predictions_lgb) + (optimal_weights[1] * test_predictions_xgb), 
    0.0, 
    1.0
)

submission_payload = pd.DataFrame({
    'Index': submission_index,
    'demand': final_test_predictions
})

submission_payload.to_csv("submission_v9.csv", index=False)

final_r2 = max(0, 100 * r2_score(
    y_train[validated_indices], 
    (optimal_weights[0] * oof_predictions_lgb[validated_indices] + optimal_weights[1] * oof_predictions_xgb[validated_indices])
))

print("\nPipeline execution complete.")
print(f"Validated Terminal R2 Score: {final_r2:.4f}")
print("Output generated: submission_v9.csv")


Pipeline execution complete.
Validated Terminal R2 Score: 81.9311
Output generated: submission_v9.csv
